# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. We'll examine ordered logistic regression outputs, socio-demographic predictors, and household adoption behaviors in rangeland management practices across Northern Kenya.

---
### Dataset Source
The dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

This schema provides metadata, structure, and access to data files, fields, and record sets. All references to entities (record sets, fields, columns) will use their `@id` values for clarity and reproducibility.


In [ ]:
# Ensure mlcroissant is installed in your environment
!pip install mlcroissant

## 1. Data Loading

We'll load the dataset metadata and initialize the `mlcroissant.Dataset` class using the Croissant schema URL. This step provides access to metadata and enables exploration of available record sets and fields.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (use object properties, not dict subscripting)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Dataset @id: {dataset.metadata.id}")
print(f"Published: {dataset.metadata.date_published}")
print(f"License: {dataset.metadata.license}")


## 2. Data Overview

Next, let's review the available record sets and fields, referencing their `@id` for navigation and extraction.

The dataset may contain several record sets—each representing tabular or structured data chunks (e.g., survey responses, regression results). We'll scan and list them, then list their fields (columns).

**Note:** All identifiers are referenced via their `@id` as per the Croissant specification.


In [ ]:
# List all available record sets and their fields
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in the metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet Name: {rs.name}")
        print(f"RecordSet @id: {rs.id}")
        print("Fields:")
        for field in rs.fields:
            print(f"  - {field.name} (@id: {field.id}, Data type: {field.data_type})")
        print("---")

## 3. Data Extraction

We'll load data from available record sets into Pandas DataFrames for analysis. Use the record set and field `@id` from the overview above.

We'll extract all record sets found, dynamically using their `@id`.


In [ ]:
dataframes = {}
record_set_ids = []

# Collect record set @ids
for rs in dataset.metadata.record_sets:
    record_set_ids.append(rs.id)

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame for record set @id: {record_set_id} (rows: {len(records)})")

# Preview columns and sample rows of the first record set (if available)
if record_set_ids:
    rs = record_set_ids[0]
    print(f"Columns for record set {rs}: ", dataframes[rs].columns.tolist())
    display(dataframes[rs].head())

## 4. Exploratory Data Analysis (EDA)

We'll process and analyze the extracted records—filtering, normalizing, and grouping data, using the relevant fields referenced by their `@id`.

Assuming the main record set contains a numeric field such as 'log likelihood' (as described in the metadata), let's demonstrate filtering, normalization, and grouping operations on this field.

All field names and group keys will be referenced by their `@id`.


In [ ]:
# Choose the first record set for EDA
rs_id = record_set_ids[0] if record_set_ids else None

# Try to identify a numeric field—example: log likelihood or coefficient
numeric_field_id = None
group_field_id = None

if rs_id:
    df = dataframes[rs_id]
    # Look for likely numeric fields
    for field in dataset.metadata.record_sets[0].fields:
        if any(term in field.name.lower() for term in ['log', 'coefficient', 'std', 'p-value']):
            numeric_field_id = field.id
            break
    # Look for a grouping field (e.g., gender, ward, intervention type)
    for field in dataset.metadata.record_sets[0].fields:
        if any(term in field.name.lower() for term in ['gender', 'ward', 'intervention', 'category']):
            group_field_id = field.id
            break

    if numeric_field_id and numeric_field_id in df.columns:
        # Filter rows where numeric field > threshold
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id} (mean values):")
            display(grouped_df.head())
    else:
        print("No numeric field identified or available for EDA.")
else:
    print("No record set available for EDA.")

## 5. Visualization

Let's visualize numeric field distributions or relationships between key predictors using matplotlib or seaborn. All axis labels should be referenced as their `@id`.

We'll create a histogram of the numeric field (if available) and a boxplot grouped by a categorical field.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if rs_id and numeric_field_id and numeric_field_id in dataframes[rs_id].columns:
    df = dataframes[rs_id]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by grouping field (if available)
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} Distribution by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

In this notebook, we've:
- Loaded the FAIR^2 dataset using `mlcroissant` and referenced the data structure via `@id` values for robust data handling.
- Reviewed dataset record sets, fields, and extracted multiple tables.
- Performed basic filtering, normalization, and grouping on ordered logistic regression predictors.
- Visualized key data distributions to facilitate analysis of rangeland knowledge adoption in Northern Kenya.

The FAIR^2 dataset provides a unique, well-structured basis for policy analysis, community planning, and further research into climate adaptation and gender-inclusive practices.

Feel free to extend the EDA and visualizations to uncover deeper insights!